# Inqaz AI - Car Crash Detection Pipeline

This notebook demonstrates the complete end-to-end pipeline for our Computer Vision project:
1. Dataset Loading & Exploration
2. Preprocessing (Resize, Normalize, Augment, Split)
3. Model Architecture (Scratch CNN + Transfer Learning)
4. Training & Evaluation
5. Results Comparison

## 1. Imports & Setup

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory
from PIL import Image
import glob

# Add src to path so we can import our modules
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU Available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 2. Dataset Exploration

Our dataset contains **3,000 real images** sourced from Kaggle, split into two classes:
- `crash/` — Images of car accidents
- `normal/` — Images of normal traffic

In [ ]:
data_dir = os.path.join(os.getcwd(), 'data', 'raw')

crash_dir = os.path.join(data_dir, 'crash')
normal_dir = os.path.join(data_dir, 'normal')

crash_count = len(os.listdir(crash_dir))
normal_count = len(os.listdir(normal_dir))

print(f'Crash images:  {crash_count}')
print(f'Normal images: {normal_count}')
print(f'Total images:  {crash_count + normal_count}')
print(f'Class balance:  {crash_count/(crash_count+normal_count)*100:.1f}% / {normal_count/(crash_count+normal_count)*100:.1f}%')

In [ ]:
# Visualize sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample Images from Dataset', fontsize=16)

crash_files = glob.glob(os.path.join(crash_dir, '*.jpg'))[:5]
normal_files = glob.glob(os.path.join(normal_dir, '*.jpg'))[:5]

for i, f in enumerate(crash_files):
    img = Image.open(f)
    axes[0][i].imshow(img)
    axes[0][i].set_title('Crash')
    axes[0][i].axis('off')

for i, f in enumerate(normal_files):
    img = Image.open(f)
    axes[1][i].imshow(img)
    axes[1][i].set_title('Normal')
    axes[1][i].axis('off')

plt.tight_layout()
plt.show()

## 3. Preprocessing Pipeline

Our preprocessing applies the following steps:
1. **Resize** all images to 224x224 pixels
2. **Normalize** pixel values to [-1.0, 1.0] using `Rescaling(1/127.5, offset=-1.0)` (required by MobileNetV2)
3. **Data Augmentation** (Random Flip, Rotation 20%, Zoom 20%) — training only
4. **Split**: 70% Train / 15% Validation / 15% Test

In [ ]:
from preprocess import get_data_generators

train_ds, val_ds, test_ds = get_data_generators(data_dir)

print(f'Train batches:      {tf.data.experimental.cardinality(train_ds).numpy()}')
print(f'Validation batches: {tf.data.experimental.cardinality(val_ds).numpy()}')
print(f'Test batches:       {tf.data.experimental.cardinality(test_ds).numpy()}')

In [ ]:
# Verify normalization range
for images, labels in train_ds.take(1):
    print(f'Batch shape: {images.shape}')
    print(f'Label shape: {labels.shape}')
    print(f'Pixel range: [{images.numpy().min():.2f}, {images.numpy().max():.2f}]')
    print(f'Expected:    [-1.00, 1.00]')

## 4. Model Architectures

### 4.1 Scratch CNN
A custom CNN built layer-by-layer: Conv2D → BatchNorm → ReLU → MaxPooling → Dense → Sigmoid

In [ ]:
from models.cnn_scratch import build_scratch_cnn

scratch_model = build_scratch_cnn()
scratch_model.summary()

### 4.2 Transfer Learning (MobileNetV2)
Pre-trained MobileNetV2 backbone with a manually coded custom head:
GlobalAveragePooling2D → Dense(256) → BatchNorm → Dropout(0.5) → Dense(128) → Dropout(0.3) → Sigmoid

In [ ]:
from models.transfer_learning import build_transfer_learning_model

tl_model, base_model = build_transfer_learning_model()
tl_model.summary()

## 5. Training Results

Models were trained using `src/train.py`. Below we load and display the saved training curves.

In [ ]:
results_dir = os.path.join(os.getcwd(), 'results')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scratch CNN curves
scratch_acc = os.path.join(results_dir, 'Scratch_CNN_accuracy.png')
scratch_loss = os.path.join(results_dir, 'Scratch_CNN_loss.png')

if os.path.exists(scratch_acc):
    img = Image.open(scratch_acc)
    axes[0].imshow(img)
    axes[0].set_title('Scratch CNN - Accuracy')
    axes[0].axis('off')

if os.path.exists(scratch_loss):
    img = Image.open(scratch_loss)
    axes[1].imshow(img)
    axes[1].set_title('Scratch CNN - Loss')
    axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Transfer Learning curves
tl_acc = os.path.join(results_dir, 'Transfer_Learning_Finetuned_accuracy.png')
tl_loss = os.path.join(results_dir, 'Transfer_Learning_Finetuned_loss.png')

if os.path.exists(tl_acc):
    img = Image.open(tl_acc)
    axes[0].imshow(img)
    axes[0].set_title('Transfer Learning - Accuracy')
    axes[0].axis('off')

if os.path.exists(tl_loss):
    img = Image.open(tl_loss)
    axes[1].imshow(img)
    axes[1].set_title('Transfer Learning - Loss')
    axes[1].axis('off')

plt.tight_layout()
plt.show()

## 6. Evaluation Results

### Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

scratch_cm = os.path.join(results_dir, 'Scratch_CNN_confusion_matrix.png')
tl_cm = os.path.join(results_dir, 'Transfer_Learning_confusion_matrix.png')

if os.path.exists(scratch_cm):
    axes[0].imshow(Image.open(scratch_cm))
    axes[0].set_title('Scratch CNN')
    axes[0].axis('off')

if os.path.exists(tl_cm):
    axes[1].imshow(Image.open(tl_cm))
    axes[1].set_title('Transfer Learning')
    axes[1].axis('off')

plt.suptitle('Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.show()

### ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

scratch_roc = os.path.join(results_dir, 'Scratch_CNN_roc_curve.png')
tl_roc = os.path.join(results_dir, 'Transfer_Learning_roc_curve.png')

if os.path.exists(scratch_roc):
    axes[0].imshow(Image.open(scratch_roc))
    axes[0].set_title('Scratch CNN - ROC')
    axes[0].axis('off')

if os.path.exists(tl_roc):
    axes[1].imshow(Image.open(tl_roc))
    axes[1].set_title('Transfer Learning - ROC')
    axes[1].axis('off')

plt.suptitle('ROC / AUC Curves', fontsize=14)
plt.tight_layout()
plt.show()

### Classification Reports

In [ ]:
for name in ['Scratch_CNN', 'Transfer_Learning']:
    report_path = os.path.join(results_dir, f'{name}_report.txt')
    if os.path.exists(report_path):
        print(f'=== {name} ===')
        with open(report_path) as f:
            print(f.read())
        print()

## 7. Model Comparison Table

| Metric | Scratch CNN | Transfer Learning (MobileNetV2) |
|--------|-------------|----------------------------------|
| Architecture | Custom 4-block CNN | MobileNetV2 + Custom Head |
| Trainable Params | ~2.5M | ~1,280 (head) + unfrozen layers |
| Pooling Strategy | Flatten | GlobalAveragePooling2D |
| Fine-tuning | N/A | 2-phase (frozen → unfreeze top 20) |
| Training Epochs | Up to 20 (EarlyStopping) | Up to 10+10 (EarlyStopping) |
| Normalization | [-1, 1] | [-1, 1] (MobileNetV2 standard) |

*Actual accuracy, F1, and timing numbers are filled in from the classification reports above after training.*

## 8. Conclusion

- Transfer Learning with MobileNetV2 outperforms the Scratch CNN due to pre-trained ImageNet features.
- `GlobalAveragePooling2D` was critical — replacing `Flatten` reduced parameters by 99% and eliminated overfitting.
- Proper normalization to `[-1, 1]` was essential for MobileNetV2 compatibility.
- Data quality validation (visual audit) proved as important as architecture design.
- The model is deployed as a live Streamlit dashboard (`app.py`) for real-time crash detection.